In [ ]:


import pandas as pd
import numpy as np
import os

from xgboost import XGBRegressor

# =========================
# 1. LOAD DATA
# =========================

df = pd.read_csv(os.path.join("..", "DATA", "data", "final_dataset.csv"))

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

# =========================
# 2. TIME FEATURES
# =========================

df["hour"]         = df["time"].dt.hour
df["day_of_week"]  = df["time"].dt.dayofweek
df["month"]        = df["time"].dt.month
df["weekend_flag"] = df["day_of_week"].isin([5, 6]).astype(int)

# Cyclical encoding — consistent with analysis_v2.py
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# =========================
# 3. LAG + ROLLING FEATURES
#    FIX: .shift(1) BEFORE .rolling() prevents current row
#    from leaking into its own feature
# =========================

df["price_lag_1"]   = df["price"].shift(1)
df["price_lag_24"]  = df["price"].shift(24)
df["price_lag_168"] = df["price"].shift(168)   # same hour last week

# ✅ Leakage-free rolling features
df["price_rolling_24"]  = df["price"].shift(1).rolling(24).mean()
df["price_rolling_168"] = df["price"].shift(1).rolling(168).mean()

# =========================
# 4. DOMAIN FEATURES
# =========================

df["residual_load"]   = df["load"] - df["wind"] - df["solar"]
df["renewable_share"] = (df["wind"] + df["solar"]) / df["load"].replace(0, np.nan)

# Price spike: based on PAST expanding quantile — no leakage
df["price_spike"] = (
    df["price"].shift(1) > df["price"].shift(1).expanding().quantile(0.90)
).astype(int)

df = df.dropna()
df = df.reset_index(drop=True)

# =========================
# 5. FEATURE SET
#    Must match analysis_v2.py exactly
# =========================

FEATURES = [
    "load", "wind", "solar",
    "residual_load", "renewable_share",   # ← this line
    "price_lag_1", "price_lag_24", "price_lag_168",
    "price_rolling_24", "price_rolling_168",
    "hour_sin", "hour_cos",
    "day_of_week", "month", "weekend_flag",
    "price_spike"
]

X = df[FEATURES]
y = df["price"]

# Strict time-based split — same 80/20 as analysis notebook
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

# =========================
# 6. TRAIN MODEL
# =========================

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

xgb_model.fit(X_train, y_train)

# Predict on full dataset for dashboard display
# (Power BI shows all rows — this is fine for visualization)
df["predicted_price"] = xgb_model.predict(X)

# Naive baseline prediction for comparison in dashboard
df["naive_predicted_price"] = df["price_lag_1"]

# Train vs Test flag — very useful in Power BI to filter/color by split
df["train_test_flag"] = "train"
df.loc[df.index >= split_index, "train_test_flag"] = "test"

# =========================
# 7. BUSINESS FEATURES
#    FIX: price_volatility_24h also needs .shift(1) to avoid leakage
# =========================

# ✅ Leakage-free volatility
df["price_volatility_24h"] = df["price"].shift(1).rolling(24).std()

# Price deviation from rolling average
df["price_deviation"] = df["price"] - df["price_rolling_24"]

# Spike flag (30% above 24h rolling average — uses already-fixed rolling_24)
df["spike_flag"] = (df["price"] > df["price_rolling_24"] * 1.3).astype(int)

# Forecast error (only meaningful on test set, but kept for all rows)
df["forecast_error"]    = df["price"] - df["predicted_price"]
df["absolute_error"]    = df["forecast_error"].abs()
df["naive_error"]       = (df["price"] - df["naive_predicted_price"]).abs()

# Did ML beat the naive baseline for this row?
df["ml_beats_naive"] = (df["absolute_error"] < df["naive_error"]).astype(int)

# Renewable generation total (useful Power BI metric)
df["total_renewable"] = df["wind"] + df["solar"]

# =========================
# 8. PRINT TEST SET METRICS
# =========================

test_df = df[df["train_test_flag"] == "test"]

print("=" * 45)
print("TEST SET METRICS (Power BI export)")
print("=" * 45)
print(f"  Naive Baseline MAE : {test_df['naive_error'].mean():.3f} €/MWh")
print(f"  XGBoost MAE        : {test_df['absolute_error'].mean():.3f} €/MWh")
print(f"  ML beats naive on  : {test_df['ml_beats_naive'].mean()*100:.1f}% of hours")
print("=" * 45)

# =========================
# 9. CLEAN & EXPORT
# =========================

df = df.dropna()

output_path = "powerbi_final_dataset.csv"
df.to_csv(output_path, index=False)

print(f"\n✅ Dataset ready for Power BI: {output_path}")
print(f"   Shape: {df.shape}")
print(f"\nColumns exported:")
for col in df.columns:
    print(f"   {col}")

TEST SET METRICS (Power BI export)
  Naive Baseline MAE : 6.991 €/MWh
  XGBoost MAE        : 6.497 €/MWh
  ML beats naive on  : 51.3% of hours

✅ Dataset ready for Power BI: powerbi_final_dataset.csv
   Shape: (9990, 30)

Columns exported:
   time
   load
   wind
   solar
   price
   hour
   day_of_week
   month
   weekend_flag
   hour_sin
   hour_cos
   price_lag_1
   price_lag_24
   price_lag_168
   price_rolling_24
   price_rolling_168
   residual_load
   renewable_share
   price_spike
   predicted_price
   naive_predicted_price
   train_test_flag
   price_volatility_24h
   price_deviation
   spike_flag
   forecast_error
   absolute_error
   naive_error
   ml_beats_naive
   total_renewable
